# Investigate Thematic Divergence

This notebook focussed on occurences of semantic divergence between the masked 'machine' token and the 'predictions'. We filtered predictions to only include sentences where the 'machine' is confused with another thematic area. This notebook focusses on the human-machine confusion. 

We there focus in human-type predictions, i.e. occurences where the LMs suggest as human-word as a filler for the machine token. 

For more information on the words we included, inspect `250_freq_pred_KB_edit.txt` and the `1-compute-gradients.ipynb`.


## What can this tell us?

We look at occurences where BERT confused the machine with human words.

We want to understand the divergence, and gauge, which **semantic** and **syntactic** patterns trigger the model towards 'human'-like predictions.

We can look more broadly which words exert push a pull related to (or away from) human or machine agency.

We only look at a smaller subset of examples, based on the existing top 10 predictions recorded in the spreadsheet created by Mariona.

We try to improve and add a few more features:
- a comparative analysis of newspapers and books, i.e. using both newspaper and book data as well as the model trained on these collections
- a more refined syntactic analysis, which includes the syntactic relation of the context token to the masked token

# On Terminology



### A thematic analysis, contrasting two concepts.

We want to understand the semantic intermingling of two distinct concepts, more specifically how the machine are related to, or portrayed as, 'human'-like entities. In this scenario, 'machine' is the TargetMaskedToken (the token that actually appeared in the text) and 'human' is the **predicted contrastive theme**, i.e. a set of thematically related words, predicted by BERT in the position of the masked machine token.

### Target vs. Contrastive Theme

One of the concepts is the **"target concept"**, to other one the **"contrastive concept"**. Each analysis starts with a target concept or token, or `TargetMaskedToken`, this is the starting point of the analysis, for example the word 'machine'. This tokens has two associated datasets. 
    - a TargetMaskedToken maps onto a set of workds that capture the target concept
    - sentences which contain the masked token (`df_target_sent`)
    - contribution obtained by from the integrated gradients method (abbr "ig", `df_target_ig`) 

The `TargetMaskedToken` related to a `ContrastiveTheme`, in this case 'human' like entitites which we filtered from BERT predictions. 

To understand language model predictions, we study both the `TargetMaskedToken` and the `ContrastiveTheme` in two scenarios: **observed** and **counterfactual**. 

 Observed refers to scenarios in which the target of constrastive token appears in the sentences. We than look what tokens contribute to predicting the actual word use. 
 
 In counterfactual scenarios, we measure which words contributed to the ''wrong'' predictions, i.e. what caused BERT to confuse machines with human entities?

## Setting things up and loading the data

In [ ]:
import pandas as pd
import json
from tqdm import tqdm
from explain import *
from pathlib import Path

In [ ]:
collection,genre_suffix = 'blb',''
if collection == 'blb':
  genre_suffix = '_with_genre'

TargetMaskedToken = 'machine' # the token to be masked in the target sentence

dataPath = 'input_data' # change to '.' when working in colab 
processedFolder = 'gradient_data' # change '.' when working in colab
predCol = "pred_bert_1760_1900"
resultType = 'pred_kw_filtered' # pred | pred_kw_filter

print(f"This analysis focuses on '{TargetMaskedToken}'.")

In [ ]:
# load the original sentences with the predicted tokens
df_sent = pd.read_csv(f'{dataPath}/{collection}_{TargetMaskedToken}{genre_suffix}_{resultType}.tsv', index_col=0, sep='\t').reset_index(drop=True)
print(f'We have {df_sent.shape[0]} sentences for the target token {TargetMaskedToken} in the {collection} collection.')
df_ig = pd.read_csv(f'{processedFolder}/results_{collection}_{TargetMaskedToken}_{resultType}_processed.csv', index_col=0 )
print(f'We have {df_ig.shape[0]} explanations for the target token {TargetMaskedToken}.')


In [ ]:
# check if ids are aligned between gradient data and sentences
print(df_ig['id'].nunique(), df_sent.shape[0])
print(df_ig[df_ig['id'] == 0].Token, df_sent.iloc[0].currentSentence) 

# Pushing towards Humanity

To find our which contextual element push the prediction to towards "human" type entitites, we start with the most generic type of question, just looking at all the all the words within the sentence across all the different (human-type) predictions.

Let's see which contextual tokens push BLERT to predicting human-like entities. Please note that the analysis below repeats sentencess, we look at all sentences and all the filtered keywords.

In [ ]:

targetTokens = ['machine','machines']
df_comparisonConcept = df_ig[
    (~df_ig['Target'].isin(targetTokens)) # we exclude the target token itself, as we are interested in other tokens that are predictive of the contrastive concept
                ].groupby('Token').agg(
                        count=('id', 'count'),identifiers=('id', set),avg_score=('Score', 'mean')
                    ).reset_index()


In [ ]:
df_ig[['Target_position','Target']].drop_duplicates()

In [ ]:
# please note that this repeats sentences, this acros all sentences with all the filtered keywords
min_count = 10
df_result = df_comparisonConcept[df_comparisonConcept['count'] >= min_count].sort_values(by='avg_score', ascending=False)
df_result.head(10)

# Explore Subthemes

Ok, the above is maybe not very helpful. Let's look at subthemes, e.g. 'child' or 'woman' related terms, both together as well as individually?

In [ ]:
wordList = ['child','children','boy','girl', 'boys', 'girls','baby','babies']

df_comparisonConcept = df_ig[
    (df_ig['Target'].isin(wordList)) # we exclude the target token itself, as we are interested in other tokens that are predictive of the contrastive concept
                ].groupby('Token').agg(
                        count=('id', 'count'),identifiers=('id', set),avg_score=('Score', 'mean')
                    ).reset_index()


In [ ]:

min_count = 10
df_result = df_comparisonConcept[df_comparisonConcept['count'] >= min_count].sort_values(by='avg_score', ascending=False)
df_result.head(10)

In [ ]:
pd.set_option('display.max_colwidth', 200)
df_sent.iloc[list(df_result.loc[6841].identifiers)].currentSentence

# Zoom in on examples

In [ ]:
modelName = "Livingwithmachines/bert_1760_1900"
explainer = MaskedLMExplainer(model_name=modelName, device=pick_device())

In [ ]:
idx = 8257
sentence = df_sent.iloc[idx].maskedSentence
targets =list(set(df_ig[df_ig['id'] == idx].Target.unique()).intersection(set(wordList)))

In [ ]:
print(f"Sentence: {sentence}")
print(f"Targets: {targets}")

In [ ]:
# select target
target = targets[0] 

In [ ]:

highlight_context_tokens(explainer, sentence, target=target, word_agg="mean")

Rank the sentences by decreasing value of the context word on the predicted word.

In [ ]:
target = 'girl'
context_token = 'kiss'
ordered_idx = df_ig[(df_ig['Target']== target) & ((df_ig['Token']==context_token))].sort_values(by='Score', ascending=False)['id'].values

# add the reverse kiss when girl is writtien

In [ ]:
df_sent.iloc[ordered_idx].currentSentence

In [ ]:
sentence = df_sent.iloc[18268].maskedSentence
highlight_context_tokens(explainer, sentence, target=target, word_agg="mean")

In [ ]:
# integrate cluster scores as a proximi for convergence
# confidently of confusedly wrong about the prediction/distrubtion of probability mass

## LLM interrogation of a result set

In [ ]:
from explain.gpt_annotation import run_gpt_preannotation
OPENAI_API_KEY = ''  # leave blank to use environment variable OPENAI_API_KEY
OPENAI_MODEL = 'gpt-4o'
OPENAI_MAX_WORKERS = 5

COT_SYSTEM_PROMPT = '''You are an expert historian reading excerpts from 19th-century British books.
We want to explore the idea that nineteenth‑century discourse often treats humans, machines, animals, and slaves as interchangeable labouring entities within a shared conceptual field.

Analyse the sentences, and assess to what extent they portray machine as being "alive" or "human-like". 
Do the sentences express anthropomorphism (machines treated as alive) or technomorphism (humans described as machines)?
Consider the context, the actions described, and any anthropomorphic or technomorphic language used. 

Think step-by-step, then output ONLY a JSON object in this exact format (no markdown, no prose, no backticks):
{"reasoning": "<chain-of-thought reasoning>", "label": "anthropomorphic", "technomorphic", or "none"}
'''


In [ ]:
df_sample = pd.DataFrame(df_sent.iloc[list(df_result.loc[6841].identifiers)].currentSentence)
df_sample.rename(columns={'currentSentence':'sentence'}, inplace=True)

df_sample.reset_index(drop=True)
df_sample['label'] = ''
df_sample['reasoning'] = ''
df_sample.head(10)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from explain.gpt_annotation import run_gpt_preannotation

summary = run_gpt_preannotation(
    df_sample,
    system_prompt=COT_SYSTEM_PROMPT,
    api_key=OPENAI_API_KEY,
    model=OPENAI_MODEL,
    n=0,
    max_workers=OPENAI_MAX_WORKERS,
)

In [ ]:
df_sample

In [ ]:
from explain.gpt_annotation import run_gpt_preannotation
OPENAI_API_KEY = ''  # leave blank to use environment variable OPENAI_API_KEY
OPENAI_MODEL = 'gpt-4o'
OPENAI_MAX_WORKERS = 5

COT_SYSTEM_PROMPT = '''You are an average person living and reading in the 19th century.

Analyse the sentences, focuss on the [MASKED] token. What word would you expect to be in the [MASKED] position, based on the context of the sentence?

Think step-by-step, then output ONLY a JSON object in this exact format (no markdown, no prose, no backticks):
{"reasoning": "<chain-of-thought reasoning>", "label": "<expected_word>"}
'''


In [ ]:
df_sample = pd.DataFrame(df_sent.iloc[list(df_result.loc[6841].identifiers)].maskedSentence)
df_sample.rename(columns={'maskedSentence':'sentence'}, inplace=True)

df_sample.reset_index(drop=True)
df_sample['label'] = ''
df_sample['reasoning'] = ''

In [ ]:
from explain.gpt_annotation import run_gpt_preannotation

summary = run_gpt_preannotation(
    df_sample,
    system_prompt=COT_SYSTEM_PROMPT,
    api_key=OPENAI_API_KEY,
    model=OPENAI_MODEL,
    n=0,
    max_workers=OPENAI_MAX_WORKERS,
)

In [ ]:
df_sample

## Temporal Filtering

In [ ]:
indices = df_sent[df_sent['date'].between(1850, 1900)].index

In [ ]:
targetTokens = ['machine','machines']
df_comparisonConcept = df_ig[
    (~df_ig['Target'].isin(targetTokens)) & \
    (df_ig['id'].isin(indices))
                ].groupby('Token').agg(
                        count=('id', 'count'),identifiers=('id', set),avg_score=('Score', 'mean')
                    ).reset_index()


In [ ]:
min_count = 50
df_comparisonConcept[df_comparisonConcept['count'] >= min_count].sort_values(by='avg_score', ascending=False).head(10)

# Temporal trends

In [ ]:
df_ig_temporal = df_ig.merge(df_sent[['date']], left_on='id', right_index=True, how='left').fillna(-1)
df_ig_temporal['decade'] = (df_ig_temporal['date']//10)*10
df_ig_temporal.head()

In [ ]:
wordList = ['woman','women']#['boy','boys','girl','girls','child','children','baby','babies']
time_unit = 'decade' #| 'date'
by_time = df_ig_temporal[df_ig_temporal.Token.isin(wordList)].groupby(time_unit).agg(
    count=('id', 'count'),avg_score=('Score', 'mean')).reset_index()
by_time[by_time[time_unit] > 1840].plot(x=time_unit, y='avg_score', kind='line', title='Average score for tokens "child" and "children" over years')

In [ ]:
by_time[by_time[time_unit] > 1840].plot(x=time_unit, y='count', kind='line', title='Average score for token "child" over decades')

In [ ]:
display(df_ig[(df_ig['id'] == 3) & (df_ig['Target'] == 'woman')])

# Syntact Analysis

In [ ]:
# look only at the tokens that are inside the [MASK] constituent
by_token = df_ig[~df_ig.mask_syntax_relation.isnull()].groupby(['Target','Token']).agg(count=('id', 'count'),identifiers=('id', list),avg_score=('Score', 'mean')).reset_index()

# look at the tokens with a specific relation to the [MASK] constituent
by_token = df_ig[df_ig.mask_constituent_relation.isin(['nsubjv','nsubj^'])].groupby(['Target','Token']).agg(count=('id', 'count'),identifiers=('id', list),avg_score=('Score', 'mean')).reset_index()


In [ ]:
min_count = 0
comparison_tokens = ['child','children']
target_tokens = ['machine','machines']

comparison = by_token[(by_token['Target'].isin(comparison_tokens)) & \
         (by_token['count'] >= min_count )
         ]
target = by_token[(by_token['Target'].isin(target_tokens)) & \
         (by_token['count'] >= min_count )
         ]
print(comparison.shape[0], target.shape[0])

In [ ]:
comparison = comparison.merge(target, on='Token', suffixes=('_kw', '_t'))
comparison['diff'] = comparison['avg_score_kw'] - comparison['avg_score_t']
comparison.sort_values(by='diff', ascending=False).head(20)


In [ ]:
context_tokens = ['factory']
target_tokens = ['child','children']

result_sentences = df_ig[(df_ig.id.isin(comparison[comparison.Token.isin(context_tokens)].identifiers_kw.values[0])) &
              df_ig['Target'].isin(target_tokens)
                                                ].groupby('id')['Token'].apply(
                                                    lambda x: ' '.join(x)).values

In [ ]:
result_sentences

# Fin.